<a href="https://colab.research.google.com/github/dan-zeman/belo-horizonte/blob/main/Udapi2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Universal Dependencies with `udapi-python`

This is the second session with the `udapi-python` library. As we start with a blank virtual machine, we must start again with installing Udapi and downloading the treebank data.

## 1. Install `udapi`

First, we need to install the `udapi` Python library. This can be done using `pip`.

In [1]:
%%bash
# (Preceding a single line with ! makes that line interpreted by shell instead of python.
# Inserting %%bash at the beginning of the cell makes the whole cell interpreted by bash.)
pip install --upgrade udapi
udapy -h

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 2.5 MB/s eta 0:00:00
usage: udapy [optional_arguments] scenario

udapy - Python interface to Udapi - API for Universal Dependencies

Examples of usage:
  udapy -s read.Sentences udpipe.En < in.txt > out.conllu
  udapy -T < sample.conllu | less -R
  udapy -HAM ud.MarkBugs < sample.conllu > bugs.html

positional arguments:
  scenario              A sequence of blocks and their parameters.

options:
  -h, --help            show this help message and exit
  -q, --quiet           Warning, info and debug messages are suppressed. Only fatal errors are reported.
  -v, --verbose         Warning, info and debug messages are printed to the STDERR.
  -s, --save            Add write.Conllu to the end of the scenario
  -T, --save_text_mode_trees
                        Add write.TextModeTrees color=1 to the end of the scenario
  -H, --save_html       Add write.TextModeTreesHtml color=1 to the end of the scenario
  -A, --save_all_attributes
 

## 2. Download and Load a UD Treebank

Download one or more UD treebanks from GitHub. Feel free to add downloads of other treebanks you are interested in. See https://universaldependencies.org/ for the list of available treebanks. Besides the UD homepage, you can also try https://universaldependencies.org/languages.html, where you can filter languages by family and genus.

Smart tip: If you want to work with your own data instead, click on the Files icon in the left pane of Colab, create a treebank folder on the local disk and upload your CoNLL-U file(s) there. Then you will be able to say that this is the treebank you want to load and process.

In [2]:
%%bash
rm -rf UD_*
git clone https://github.com/UniversalDependencies/UD_Portuguese-Porttinari.git

Cloning into 'UD_Portuguese-Porttinari'...


Now we will access the treebank data from Python. The `udapi.core.document.Document` class is used to load the treebank. You can edit the code cell below and specify the treebank you want to work with (it must be one of the treebanks you downloaded above).

In [25]:
import glob
from udapi.core.document import Document

# Read the CoNLL-U file.
###############################################################################
# TODO: Change this variable to explore different datasets.
treebank = 'UD_Portuguese-Porttinari' # Student can modify this line.

# List all .conllu files in the specified treebank folder.
conllu_files = glob.glob(f"{treebank}/*.conllu")
print(f"Found {len(conllu_files)} CoNLL-U files in {treebank}:")
for file in sorted(conllu_files):
    print(file)
print(f"Loading {treebank}...")
# Each CoNLL-U file will be stored as one Document object.
# Note that some treebanks use the '# newdoc' comment to mark document boundaries within each file.
# That is a different notion of document!
filedocs = []
nsent = 0
for file in sorted(conllu_files):
    doc = Document(filename=file)
    filedocs.append(doc)
    nsent += len(doc.bundles)
print(f"Loaded {nsent} sentences from {treebank}.")


Found 3 CoNLL-U files in UD_Portuguese-Porttinari:
UD_Portuguese-Porttinari/pt_porttinari-ud-dev.conllu
UD_Portuguese-Porttinari/pt_porttinari-ud-test.conllu
UD_Portuguese-Porttinari/pt_porttinari-ud-train.conllu
Loading UD_Portuguese-Porttinari...
Loaded 8418 sentences from UD_Portuguese-Porttinari.


## 3. Get the Trees

In the previous session, we were collecting counts of various observations in the treebank, and displaying them in a table. But can we use Udapi also to display the trees with examples? Yes, we can!

Simple searches can be achieved by calling the so-called command-line interface of Udapi; it means that we are calling Udapi as a standalone program (which is actually spelled `udapy`, note the final letter).

In [4]:
from google.colab import files

# Run udapy as a standalone program.
# The first part of the command sends our CoNLL-U files as input to the pipeline.
# The util.Mark node='...' part will mark each node that satisfies the condition.
# The -HAM options mean that udapy will output the trees as colored HTML,
# highlighting the marked nodes. The output will contain only trees that contain
# at least one marked node.
!cat UD_Portuguese-Porttinari/*.conllu | udapy -HAM util.Mark node='node.udeprel == "expl" and node.parent.upos == "VERB"' > udapi_output.html

# Offer the file for download so that you can open it in your browser.
# We could display it here in Colab but it would not be practical because the
# file could be very long.
print(f"File 'udapi_output.html' is ready for download.")
files.download('udapi_output.html')


2026-07-23 19:35:28,496 [   INFO] run_blocks - No reader specified, using read.Conllu
2026-07-23 19:35:28,496 [   INFO] run_blocks -  ---- ROUND ----
2026-07-23 19:35:28,497 [   INFO] run_blocks - Executing block read.Conllu {}
2026-07-23 19:35:28,497 [   INFO] try_fast_load - Reading -
2026-07-23 19:35:29,302 [   INFO] run_blocks - Executing block util.Mark node=node.udeprel == "expl" and node.parent.upos == "VERB"
2026-07-23 19:35:32,149 [   INFO] run_blocks - Executing block write.TextModeTreesHtml color=1 attributes=form,lemma,upos,xpos,feats,deprel,misc marked_only=1
File 'udapi_output.html' is ready for download.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Text Mode, One Tree at a Time

We can also ask Udapi to draw the tree structure if we are using Udapi's functions from within Python. In that case, we use the function `root.draw()`.

Here we have two code cells. The first one defines a function that will find the next sentence containing at least one node with a given UPOS (`search_upos`), mark those nodes by adding a temporary attribute Mark to their annotation, and return the sentence so that it can be displayed. Further down, the cell says that our `search_upos` is `VERB`. We need to run this cell only once, as we do with most cells in the notebook.

The subsequent cell is different. We can run it many times and each time it will deliver the next example sentence, until we reach the end of the treebank (or until we run the initialization cell again).

In [26]:
def find_and_yield_sentences(filedocs):
    """
    A generator function that yields sentences (bundles) containing a node
    with the specified Universal Part-of-Speech (UPOS) tag.
    """
    for doc in filedocs:
        for bundle in doc.bundles:
            hits = 0
            # Get all nodes in the current sentence (bundle)
            nodes = bundle.get_tree().descendants
            for node in nodes:
                # Check if any node in the sentence matches the search condition.
                if node.upos == 'PRON' and node.feats['PronType'] == 'Prs':
                    node.misc['Mark'] = '<==============='
                    hits += 1
            if hits > 0:
                yield bundle # Yield the entire bundle (sentence) when found

# Example usage of the generator:
# Let's search for sentences containing a verb (VERB UPOS).
# Initialize the generator. This creates the iterator object.
# This object will maintain its state across calls to `next()`.
################################################################################
# TODO: You can replace 'VERB' with another UPOS (or modify the condition inside the generator).
verb_sentence_iterator = find_and_yield_sentences(filedocs)

print("Initialized generator to find sentences.")

Initialized generator to find sentences.


In [29]:
try:
    # Get the next sentence from the generator
    next_verb_sentence = next(verb_sentence_iterator)
    root = next_verb_sentence.get_tree()

    # Display the tree structure using root.draw()
    root.draw(attributes='form,lemma,upos,feats,deprel,misc[Mark]')

except StopIteration:
    print("No more results.")
    print("Run the initialization cell again (the one defining `verb_sentence_iterator`) to start from the beginning.")

# sent_id = FOLHA_DOC004096_SENT029
# text = Para se livrar do problema, dobrou a aposta e injetou mais R$ 2 bilhões na JBS.
─┮
 │   ╭─╼ Para para ADP _ mark 
 │   ┢─╼ se se PRON Case=Acc|Person=3|PronType=Prs expl <===============
 │ ╭─┾ livrar livrar VERB Number=Sing|Person=3|VerbForm=Inf advcl 
 │ │ │ ╭─╼ de de ADP _ case 
 │ │ │ ┢─╼ o o DET Definite=Def|Gender=Masc|Number=Sing|PronType=Art det 
 │ │ ┡─┶ problema problema NOUN Gender=Masc|Number=Sing obl 
 │ │ ╰─╼ , , PUNCT _ punct 
 ╰─┾ dobrou dobrar VERB Mood=Ind|Number=Sing|Person=3|Tense=Past|VerbForm=Fin root 
   │ ╭─╼ a o DET Definite=Def|Gender=Fem|Number=Sing|PronType=Art det 
   ┡─┶ aposta aposta NOUN Gender=Fem|Number=Sing obj 
   │ ╭─╼ e e CCONJ _ cc 
   ┡─┾ injetou injetar VERB Mood=Ind|Number=Sing|Person=3|Tense=Past|VerbForm=Fin conj 
   │ │ ╭─╼ mais mais DET PronType=Ind det 
   │ │ ┢─╼ R$ R$ SYM _ nmod 
   │ │ ┢─╼ 2 2 NUM NumType=Card nummod 
   │ ┡─┶ bilhões bilhão NOUN Gender=Masc|Number=Plur obj 
   │ │ ╭─╼ em em 

## 4. Parsing Custom Text with UDPipe

You can call the UDPipe online parsing service from within Udapi. This section allows you to provide your own text, parse it using the UDPipe online service, and integrate the resulting linguistic trees into your Colab environment for further analysis.

### Enter your text below

In [30]:
user_text_input = 'Belo Horizonte é a capital do estado brasileiro de Minas Gerais. Sua população é de 2 315 560 habitantes, segundo o censo de 2022,[4] sendo o sexto município mais populoso do país, o terceiro da Região Sudeste e o primeiro de seu estado.[9] Com uma área de aproximadamente 331 km², possui uma geografia diversificada, com morros e baixadas. Com uma distância de 716 quilômetros de Brasília, é a segunda capital de estado mais próxima da capital federal, depois de Goiânia.' # @param {type:"string"}

print(f"The text you entered:\n'{user_text_input}'")

The text you entered:
'Belo Horizonte é a capital do estado brasileiro de Minas Gerais. Sua população é de 2 315 560 habitantes, segundo o censo de 2022,[4] sendo o sexto município mais populoso do país, o terceiro da Região Sudeste e o primeiro de seu estado.[9] Com uma área de aproximadamente 331 km², possui uma geografia diversificada, com morros e baixadas. Com uma distância de 716 quilômetros de Brasília, é a segunda capital de estado mais próxima da capital federal, depois de Goiânia.'


### Select UDPipe Model

Choose the UDPipe model you want to use for parsing your text. The model will be loaded from the UDPipe online service.

In [31]:
from udapi.tool.udpipeonline import UDPipeOnline

udpipe_model = 'portuguese-porttinari-ud-2.17-251125' # @param ['portuguese-porttinari-ud-2.17-251125', 'portuguese-petrogold-ud-2.17-251125', 'portuguese-bosque-ud-2.17-251125', 'portuguese-dantestocks-ud-2.17-251125']

print(f"Selected UDPipe model: {udpipe_model}")

Selected UDPipe model: portuguese-porttinari-ud-2.17-251125


### Parse the text

The `udapi.block.udpipe.udpipeonline` block will now send your text to the UDPipe service, parse it, and return the result as a `Document` object.

In [32]:
from udapi.core.document import Document
from udapi.core.bundle import Bundle
from udapi.core.node import Node # Node is the base class for tree nodes

print("Sending text to UDPipe...")
# Create a UDPipeOnline block with the selected model
# The input should be a Document object with bundles containing the raw text

# Initialize dummy_doc with one bundle containing one tree (just the root node, with root.text set to user_text_input)
# This explicitly creates a single bundle and a single root node.
# The UDPipeOnline block will then process the raw text from this document.
dummy_doc = Document()
dummy_doc.text = user_text_input # Set the document's raw text content for UDPipeOnline
bundle = dummy_doc.create_bundle()
root = bundle.create_tree()
root.text = user_text_input # Set the root node's text content

# Instantiate and run the UDPipeOnline block
pipe = UDPipeOnline(model=udpipe_model)
pipe.process_document(dummy_doc) # The dummy_doc is modified in-place

user_input_docs = dummy_doc # Store the parsed document(s) for later use

print("Parsing completed. Parsed trees are saved in 'user_input_docs'.")
print(f"Number of sentences in the user input: {len(user_input_docs.bundles)}")

Sending text to UDPipe...
Parsing completed. Parsed trees are saved in 'user_input_docs'.
Number of sentences in the user input: 1


### Display the first parsed sentence

To confirm the parsing was successful, here's the tree structure for the first sentence from your input text.

In [33]:
if user_input_docs and len(user_input_docs.bundles) > 0:
    first_bundle = user_input_docs.bundles[0]
    first_root = first_bundle.get_tree()
    print("Tree structure of the first sentence:")
    first_root.draw(attributes='form,lemma,upos,feats,deprel,misc')
else:
    print("We did not find any sentences to display.")

Tree structure of the first sentence:
# sent_id = 1
# text = Belo Horizonte é a capital do estado brasileiro de Minas Gerais. Sua população é de 2 315 560 habitantes, segundo o censo de 2022,[4] sendo o sexto município mais populoso do país, o terceiro da Região Sudeste e o primeiro de seu estado.[9] Com uma área de aproximadamente 331 km², possui uma geografia diversificada, com morros e baixadas. Com uma distância de 716 quilômetros de Brasília, é a segunda capital de estado mais próxima da capital federal, depois de Goiânia.
─┮
 │ ╭─┮ Belo Belo PROPN _ nsubj _
 │ │ ╰─╼ Horizonte Horizonte PROPN _ flat:name _
 │ ┢─╼ é ser AUX Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin cop _
 │ ┢─╼ a o DET Definite=Def|Gender=Fem|Number=Sing|PronType=Art det _
 ╰─┾ capital capital NOUN Number=Sing root _
   │ ╭─╼ de de ADP _ case _
   │ ┢─╼ o o DET Definite=Def|Gender=Masc|Number=Sing|PronType=Art det _
   ┡─┾ estado estado NOUN Gender=Masc|Number=Sing nmod _
   │ ┡─╼ brasileiro brasileiro 